##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the \"License\");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an \"AS IS\" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: 오류 처리

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Error_handling.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

이 Colab 노트북은 Gemini API 사용 시 발생할 수 있는 일반적인 오류를 처리하는 전략을 설명합니다:

*   **일시적 오류(Transient Errors):** 네트워크 문제, 서버 과부하 등으로 인한 일시적인 장애.
*   **속도 제한(Rate Limits):** 일정 시간 내에 처리할 수 있는 요청 수 제한.
*   **타임아웃(Timeouts):** API 호출이 완료되는 데 너무 오래 걸리는 경우.

다음 두 가지 주요 접근 방식을 살펴볼 수 있습니다:

1.  **자동 재시도:** 일시적 오류로 인해 요청이 실패할 때 재시도하는 간단한 방법.
2.  **수동 백오프 및 재시도:** 재시도 동작을 더 세밀하게 제어하는 맞춤형 접근 방식.


**Gemini 속도 제한**

다양한 Gemini 모델의 기본 속도 제한은 [Gemini API 모델 문서](https://ai.google.dev/gemini-api/docs/models/gemini#model-variations)에 설명되어 있습니다. 애플리케이션에 더 높은 할당량이 필요한 경우 [속도 제한 증가 요청](https://ai.google.dev/gemini-api/docs/quota)을 고려하세요.

In [ ]:
%pip install -q -U "google-genai>=1.0.0"

### API 키 설정

다음 셀을 실행하려면 API 키를 `GOOGLE_API_KEY`라는 이름의 Colab 시크릿에 저장해야 합니다. API 키가 없거나 Colab 시크릿 생성 방법이 궁금하다면 [인증](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) 가이드를 참고하세요.

In [ ]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

### 자동 재시도

Gemini API 클라이언트 라이브러리는 일시적 오류 처리를 위한 내장 재시도 메커니즘을 제공합니다. `generate_content`, `generate_answer`, `embed_content`, `generate_content_async`와 같은 API 호출에서 `request_options` 인수를 사용하여 이 기능을 활성화할 수 있습니다.

**장점:**

* **간결함:** 최소한의 코드 변경으로 상당한 안정성 향상을 달성합니다.
* **견고함:** 추가 로직 없이 대부분의 일시적 오류를 효과적으로 처리합니다.

**재시도 동작 커스터마이징:**

[`retry`](https://googleapis.dev/python/google-api-core/latest/retry.html)에서 다음 설정을 사용하여 재시도 동작을 커스터마이징할 수 있습니다:

* `predicate`: (callable) 예외가 재시도 가능한지 여부를 결정합니다. 기본값: [`if_transient_error`](https://github.com/googleapis/python-api-core/blob/main/google/api_core/retry/retry_base.py#L75C4-L75C13)
* `initial`: (float) 첫 번째 재시도 전 초기 지연 시간(초). 기본값: `1.0`
* `maximum`: (float) 재시도 간 최대 지연 시간(초). 기본값: `60.0`
* `multiplier`: (float) 각 재시도 후 지연 시간이 증가하는 배율. 기본값: `2.0`
* `timeout`: (float) 전체 재시도 지속 시간(초). 기본값: `120.0`

In [ ]:
from google.api_core import retry

MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

prompt = "Write a story about a magic backpack."

#Built in retry support was removed from the sdk, so you need to use retry package
@retry.Retry(
    predicate=retry.if_transient_error,
)
def generate_with_retry():
  return client.models.generate_content(
      model=MODEL_ID,
      contents=prompt
  )

generate_with_retry().text

'Elara wasn’t looking for magic. She was looking for a backpack. Her old one, affectionately nicknamed “The Beast,” had finally given up the ghost, its seams ripped and its zipper permanently jammed. So, she found herself in Mrs. Willowby’s Oddity Emporium, a place smelling of mothballs and forgotten dreams.\n\nThe backpack in question was tucked away in a dusty corner, almost hidden behind a taxidermied two-headed duck. It was made of a deep indigo fabric, embroidered with silver constellations that shimmered faintly even in the dim light. It was perfect.\n\n“That one’s been here for ages,” Mrs. Willowby croaked, dusting it off with a flourish. “Nobody seems to want it.”\n\nElara didn\'t care. She paid the paltry sum, slung the backpack over her shoulder, and hurried home.\n\nThe first sign that something was amiss came the next day. Packing for school, Elara discovered the backpack was inexplicably larger inside than out. She could fit her textbooks, lunch, a bulky art project, and s

### 응답에 시간이 걸릴 때 수동으로 타임아웃 늘리기

`ReadTimeout` 또는 `DeadlineExceeded` 오류, 즉 API 호출이 기본 타임아웃(600초)을 초과하는 경우, `request_options` 인수에서 `timeout`을 정의하여 수동으로 조정할 수 있습니다.

In [ ]:
from google.genai import types
prompt = "Write a story about a magic backpack."

client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
       http_options=types.HttpOptions(
           timeout=15*60*1000
       )
    )
)  # Increase timeout to 15 minutes

GenerateContentResponse(candidates=[Candidate(content=Content(parts=[Part(video_metadata=None, thought=None, code_execution_result=None, executable_code=None, file_data=None, function_call=None, function_response=None, inline_data=None, text='Flora had always been unremarkable. Brown hair, brown eyes, perpetually lost in a book. Even her backpack, a drab canvas thing she\'d inherited from her older brother, screamed "invisible." Until, one Tuesday morning, it didn\'t.\n\nShe was rushing to catch the bus, her fingers fumbling with the zipper of the aforementioned backpack, when it refused to budge. Frustrated, she yanked harder. There was a ripping sound, but instead of canvas tearing, a shimmering, iridescent light spilled out.\n\nFlora gasped. The inside of the backpack wasn\'t canvas anymore. It was… a swirling vortex of colours, like the aurora borealis compressed into a small space. Hesitantly, she reached in. Her fingers brushed against something soft, and she pulled it out.\n\nIt

**주의:** 타임아웃을 늘리는 것이 도움이 될 수 있지만, 너무 높게 설정하면 오류 감지가 지연되고 리소스가 낭비될 수 있으므로 주의하세요.

### 오류 처리와 함께 백오프 및 재시도 수동 구현

재시도 동작과 오류 처리를 더 세밀하게 제어하려면 [`retry`](https://googleapis.dev/python/google-api-core/latest/retry.html) 라이브러리(또는 [`backoff`](https://pypi.org/project/backoff/), [`tenacity`](https://tenacity.readthedocs.io/en/latest/)와 같은 유사 라이브러리)를 사용할 수 있습니다. 이를 통해 재시도 전략을 정밀하게 제어하고 특정 유형의 오류를 다르게 처리할 수 있습니다.

In [ ]:
from google.api_core import retry, exceptions

MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

@retry.Retry(
    predicate=retry.if_transient_error,
    initial=2.0,
    maximum=64.0,
    multiplier=2.0,
    timeout=600,
)
def generate_with_retry(prompt):
    return client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )


prompt = "Write a one-liner advertisement for magic backpack."

generate_with_retry(prompt=prompt)

GenerateContentResponse(candidates=[Candidate(content=Content(parts=[Part(video_metadata=None, thought=None, code_execution_result=None, executable_code=None, file_data=None, function_call=None, function_response=None, inline_data=None, text='Unzip endless possibilities with the Magic Backpack - more space, more adventure!\n')], role='model'), citation_metadata=None, finish_message=None, token_count=None, finish_reason=<FinishReason.STOP: 'STOP'>, avg_logprobs=-0.6894590854644775, grounding_metadata=None, index=None, logprobs_result=None, safety_ratings=None)], create_time=None, response_id=None, model_version='gemini-2.0-flash', prompt_feedback=None, usage_metadata=GenerateContentResponseUsageMetadata(cache_tokens_details=None, cached_content_token_count=None, candidates_token_count=16, candidates_tokens_details=[ModalityTokenCount(modality=<MediaModality.TEXT: 'TEXT'>, token_count=16)], prompt_token_count=10, prompt_tokens_details=[ModalityTokenCount(modality=<MediaModality.TEXT: 'TE

### 재시도 메커니즘을 통한 오류 처리 테스트

오류 처리 및 재시도 메커니즘이 의도한 대로 작동하는지 검증하기 위해, 첫 번째 호출에서 의도적으로 `ServiceUnavailable` 오류를 발생시키는 `generate_content` 함수를 정의합니다. 이 설정은 재시도 데코레이터가 일시적 오류를 성공적으로 처리하고 작업을 재시도하는지 확인하는 데 도움이 됩니다.

In [ ]:
from google.api_core import retry, exceptions


@retry.Retry(
    predicate=retry.if_transient_error,
    initial=2.0,
    maximum=64.0,
    multiplier=2.0,
    timeout=600,
)
def generate_content_first_fail(prompt):
    if not hasattr(generate_content_first_fail, "call_counter"):
        generate_content_first_fail.call_counter = 0

    generate_content_first_fail.call_counter += 1

    try:
        if generate_content_first_fail.call_counter == 1:
            raise exceptions.ServiceUnavailable("Service Unavailable")

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt
        )
        return response.text
    except exceptions.ServiceUnavailable as e:
        print(f"Error: {e}")
        raise


prompt = "Write a one-liner advertisement for magic backpack."

generate_content_first_fail(prompt=prompt)

Error: 503 Service Unavailable


'Unzip the impossible with the Magic Backpack - where adventure always fits!\n'